In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd

from src.portfolio import (
    simulate_portfolio,
    calculate_portfolio_metrics,
    get_rf
)

In [3]:
df = pd.read_csv("../data/processed/panel_dataset_v1.csv")
df["date"] = pd.to_datetime(df["date"])

df.head()

,date,ticker,return_1m,index_return,target,return_lag1,ret_3m,ret_6m,vol_3m,vol_6m,excess_return,rank_ret_3m,inflation,unemployment,interest_rate
0,2010-08-01,ADS.DE,-0.033566,-0.043466,1,0.042257,-0.048492,0.140782,0.057823,0.071284,0.009900,0.10,1.7,8.9,1.0
1,2010-09-01,ADS.DE,0.130586,0.047637,0,-0.033566,-0.008688,0.121305,0.039932,0.073737,0.082949,0.25,1.6,8.5,1.0
2,2010-10-01,ADS.DE,0.032152,0.035332,1,0.130586,0.139277,0.163680,0.082155,0.083000,-0.003181,0.75,1.9,8.5,1.0
3,2010-11-01,ADS.DE,0.031790,-0.068190,0,0.032152,0.129172,0.080680,0.082618,0.071553,0.099980,0.80,1.9,8.4,1.0
4,2010-12-01,ADS.DE,0.010959,0.053501,0,0.031790,0.194528,0.185840,0.056936,0.057542,-0.042541,0.75,1.9,8.1,1.0


In [4]:
df.columns

Index(['date', 'ticker', 'return_1m', 'index_return', 'target', 'return_lag1',
       'ret_3m', 'ret_6m', 'vol_3m', 'vol_6m', 'excess_return', 'rank_ret_3m',
       'inflation', 'unemployment', 'interest_rate'],
      dtype='str')

In [5]:
market_features = [
    "return_lag1",
    "ret_3m",
    "ret_6m",
    "vol_3m",
    "vol_6m",
    "excess_return",
    "rank_ret_3m"
]

market_macro_features = market_features + [
    "inflation",
    "interest_rate",
    "unemployment"
]

In [6]:
rf_returns = simulate_portfolio(
    df=df,
    model_func=get_rf,
    features=market_macro_features,
    top_pct=0.2,
    start_test_year=2015
)

rf_returns.head()

date
2015-01-01    0.103030
2015-02-01    0.069697
2015-03-01   -0.001701
2015-04-01   -0.015275
2015-05-01   -0.010587
Name: portfolio_return, dtype: float64

In [7]:
metrics = calculate_portfolio_metrics(rf_returns)

print(metrics)

{'cumulative_return': np.float64(3.141001031093329), 'annualized_volatility': np.float64(0.20950695440888212), 'sharpe_ratio': np.float64(0.7963871770717077), 'max_drawdown': np.float64(-0.27623673374068847)}


In [8]:
df["target"] = ...

In [9]:
df["return_1m"] = ...

In [10]:
rf_market_returns = simulate_portfolio(
    df=df,
    model_func=get_rf,
    features=market_features,
    top_pct=0.2,
    start_test_year=2015
)

rf_market_metrics = calculate_portfolio_metrics(rf_market_returns)

print(rf_market_metrics)

ValueError: Unknown label type: unknown. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

In [11]:
df[market_features].isna().sum()

return_lag1      0
ret_3m           0
ret_6m           0
vol_3m           0
vol_6m           0
excess_return    0
rank_ret_3m      0
dtype: int64

In [12]:
df[market_macro_features].isna().sum()

return_lag1      0
ret_3m           0
ret_6m           0
vol_3m           0
vol_6m           0
excess_return    0
rank_ret_3m      0
inflation        0
interest_rate    0
unemployment     0
dtype: int64

In [13]:
df_clean_market = df.dropna(subset=market_features + ["target", "return_1m"])

rf_market_returns = simulate_portfolio(
    df=df_clean_market,
    model_func=get_rf,
    features=market_features,
    top_pct=0.2,
    start_test_year=2015
)

rf_market_metrics = calculate_portfolio_metrics(rf_market_returns)

print(rf_market_metrics)

ValueError: Unknown label type: unknown. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

In [14]:
for year in sorted(df["date"].dt.year.unique()):
    if year < 2015:
        continue

    train = df[df["date"].dt.year < year].dropna(
        subset=market_features + ["target", "return_1m"]
    )

    print(
        year,
        "n_train:", len(train),
        "target_counts:",
        train["target"].value_counts(dropna=False).to_dict()
    )

2015 n_train: 1060 target_counts: {Ellipsis: 1060}
2016 n_train: 1300 target_counts: {Ellipsis: 1300}
2017 n_train: 1540 target_counts: {Ellipsis: 1540}
2018 n_train: 1780 target_counts: {Ellipsis: 1780}
2019 n_train: 2020 target_counts: {Ellipsis: 2020}
2020 n_train: 2260 target_counts: {Ellipsis: 2260}
2021 n_train: 2500 target_counts: {Ellipsis: 2500}
2022 n_train: 2740 target_counts: {Ellipsis: 2740}
2023 n_train: 2980 target_counts: {Ellipsis: 2980}
2024 n_train: 3220 target_counts: {Ellipsis: 3220}


In [15]:
df["target"].head(20)

0     Ellipsis
1     Ellipsis
2     Ellipsis
3     Ellipsis
4     Ellipsis
5     Ellipsis
6     Ellipsis
7     Ellipsis
8     Ellipsis
9     Ellipsis
10    Ellipsis
11    Ellipsis
12    Ellipsis
13    Ellipsis
14    Ellipsis
15    Ellipsis
16    Ellipsis
17    Ellipsis
18    Ellipsis
19    Ellipsis
Name: target, dtype: object

In [16]:
df["target"].unique()

array([Ellipsis], dtype=object)

In [17]:
df = df.sort_values(["ticker", "date"])

df["future_return"] = df.groupby("ticker")["return_1m"].shift(-1)
df["future_index_return"] = df["index_return"].shift(-1)

df["target"] = (df["future_return"] > df["future_index_return"]).astype(int)

df = df.dropna(subset=["future_return", "future_index_return"])

TypeError: '>' not supported between instances of 'ellipsis' and 'float'

In [ ]:
df["target"].value_counts()

In [18]:
df["target"].head(20)

0     Ellipsis
1     Ellipsis
2     Ellipsis
3     Ellipsis
4     Ellipsis
5     Ellipsis
6     Ellipsis
7     Ellipsis
8     Ellipsis
9     Ellipsis
10    Ellipsis
11    Ellipsis
12    Ellipsis
13    Ellipsis
14    Ellipsis
15    Ellipsis
16    Ellipsis
17    Ellipsis
18    Ellipsis
19    Ellipsis
Name: target, dtype: object

In [19]:
df["target"].unique()

array([Ellipsis], dtype=object)

In [20]:
df = df.sort_values(["ticker", "date"])

# rentabilidad futura de la acción
df["future_return"] = (
    df.groupby("ticker")["return_1m"].shift(-1)
)

# rentabilidad futura del índice
df["future_index_return"] = (
    df.groupby("ticker")["index_return"].shift(-1)
)

# target binario
df["target"] = (
    df["future_return"] > df["future_index_return"]
).astype(int)

# eliminar filas sin futuro disponible
df = df.dropna(
    subset=["future_return", "future_index_return"]
)

TypeError: '>' not supported between instances of 'ellipsis' and 'float'

In [21]:
df[["return_1m", "index_return", "target"]].head(20)

,return_1m,index_return,target
0,Ellipsis,-0.043466,Ellipsis
1,Ellipsis,0.047637,Ellipsis
2,Ellipsis,0.035332,Ellipsis
3,Ellipsis,-0.068190,Ellipsis
4,Ellipsis,0.053501,Ellipsis
5,Ellipsis,0.057580,Ellipsis
6,Ellipsis,0.020131,Ellipsis
7,Ellipsis,-0.033912,Ellipsis
8,Ellipsis,0.034470,Ellipsis
9,Ellipsis,-0.063711,Ellipsis


In [22]:
df[["return_1m", "index_return", "target"]].applymap(type).head(20)

AttributeError: 'DataFrame' object has no attribute 'applymap'

In [ ]:
import numpy as np

df = df.copy()

# Sustituir Ellipsis (...) por NaN
df = df.replace({Ellipsis: np.nan})

# Convertir columnas numéricas
numeric_cols = [
    "return_1m",
    "index_return",
    "return_lag1",
    "ret_3m",
    "ret_6m",
    "vol_3m",
    "vol_6m",
    "excess_return",
    "rank_ret_3m",
    "inflation",
    "unemployment",
    "interest_rate",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Reconstruir target
df = df.sort_values(["ticker", "date"])

df["future_return"] = df.groupby("ticker")["return_1m"].shift(-1)
df["future_index_return"] = df.groupby("ticker")["index_return"].shift(-1)

df["target"] = (df["future_return"] > df["future_index_return"]).astype(int)

# Eliminar filas incompletas
df = df.dropna(subset=numeric_cols + ["future_return", "future_index_return", "target"])

In [23]:
import numpy as np

# Reemplazar Ellipsis (...) por NaN
df = df.replace({Ellipsis: np.nan})

# Columnas numéricas
numeric_cols = [
    "return_1m",
    "index_return",
    "return_lag1",
    "ret_3m",
    "ret_6m",
    "vol_3m",
    "vol_6m",
    "excess_return",
    "rank_ret_3m",
    "inflation",
    "unemployment",
    "interest_rate",
]

# Convertir a numérico
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Orden temporal
df = df.sort_values(["ticker", "date"])

# Variables futuras
df["future_return"] = (
    df.groupby("ticker")["return_1m"].shift(-1)
)

df["future_index_return"] = (
    df.groupby("ticker")["index_return"].shift(-1)
)

# Target binario correcto
df["target"] = (
    df["future_return"] > df["future_index_return"]
).astype(int)

# Eliminar filas con nulos
df = df.dropna(
    subset=numeric_cols + [
        "future_return",
        "future_index_return",
        "target"
    ]
)

print(df["target"].value_counts())

Series([], Name: count, dtype: int64)


In [24]:
df_raw = pd.read_csv("../data/processed/panel_dataset_v1.csv")
df_raw["date"] = pd.to_datetime(df_raw["date"])

df_raw.head()

,date,ticker,return_1m,index_return,target,return_lag1,ret_3m,ret_6m,vol_3m,vol_6m,excess_return,rank_ret_3m,inflation,unemployment,interest_rate
0,2010-08-01,ADS.DE,-0.033566,-0.043466,1,0.042257,-0.048492,0.140782,0.057823,0.071284,0.009900,0.10,1.7,8.9,1.0
1,2010-09-01,ADS.DE,0.130586,0.047637,0,-0.033566,-0.008688,0.121305,0.039932,0.073737,0.082949,0.25,1.6,8.5,1.0
2,2010-10-01,ADS.DE,0.032152,0.035332,1,0.130586,0.139277,0.163680,0.082155,0.083000,-0.003181,0.75,1.9,8.5,1.0
3,2010-11-01,ADS.DE,0.031790,-0.068190,0,0.032152,0.129172,0.080680,0.082618,0.071553,0.099980,0.80,1.9,8.4,1.0
4,2010-12-01,ADS.DE,0.010959,0.053501,0,0.031790,0.194528,0.185840,0.056936,0.057542,-0.042541,0.75,1.9,8.1,1.0


In [25]:
df_raw.isna().sum()

date             0
ticker           0
return_1m        0
index_return     0
target           0
return_lag1      0
ret_3m           0
ret_6m           0
vol_3m           0
vol_6m           0
excess_return    0
rank_ret_3m      0
inflation        0
unemployment     0
interest_rate    0
dtype: int64

In [26]:
df_raw.dtypes

date             datetime64[us]
ticker                      str
return_1m               float64
index_return            float64
target                    int64
return_lag1             float64
ret_3m                  float64
ret_6m                  float64
vol_3m                  float64
vol_6m                  float64
excess_return           float64
rank_ret_3m             float64
inflation               float64
unemployment            float64
interest_rate           float64
dtype: object

In [27]:
df_raw[["return_1m", "index_return", "target"]].head(20)

,return_1m,index_return,target
0,-0.033566,-0.043466,1
1,0.130586,0.047637,0
2,0.032152,0.035332,1
3,0.031790,-0.068190,0
4,0.010959,0.053501,0
5,-0.069544,0.057580,1
6,0.022203,0.020131,0
7,-0.043979,-0.033912,1
8,0.130582,0.034470,1
9,0.042379,-0.063711,1
